In [ ]:
from lxml import etree
from rapidfuzz import fuzz
from pathlib import Path
import json

# =====================================================
# UTILS
# =====================================================

def normalize(text):
    return text.lower().strip() if text else ""


# =====================================================
# 1. EXTRACTION GLOBALE DES BALISES
# =====================================================

def extract_global_tags(xml_string: str):
    root = etree.fromstring(xml_string.encode())

    ns = {"tei": root.nsmap.get(None)} if None in root.nsmap else {}

    def xp(path):
        return root.xpath(path, namespaces=ns)

    #Éléments à évaluer au niveau 2 parallèle (à commenter ou décommenter)
    #definition = xp(".//tei:def/text()" if ns else ".//def/text()")
    #traduction = xp(".//tei:cit[@type='translation']/tei:quote/text()" if ns else ".//cit[@type='translation']/quote/text()")
    #exemple = xp(".//tei:cit[@type='example']/tei:quote/text()" if ns else ".//cit[@type='example']/quote/text()")
    #renvoi = xp(".//tei:xr/text()" if ns else ".//xr/text()")
    #reference = xp(".//tei:ref/text()" if ns else ".//ref/text()")
    #encycl = xp(".//tei:seg/text()" if ns else ".//seg/text()")
    #etym = xp(".//tei:etym/text()" if ns else ".//etym/text()")
    #glose = xp(".//tei:gloss/text()" if ns else ".//gloss/text()")
    
    #Éléments à évaluer au niveau 3 parallèle (à commenter ou décommenter)
    bibl = xp(".//tei:bibl/text()" if ns else ".//bibl/text()")
    auteur = xp(".//tei:author/text()" if ns else ".//author/text()")
    oRef = xp(".//tei:oRef/text()" if ns else ".//oRef/text()")
    foreign = xp(".//tei:foreign/text()" if ns else ".//foreign/text()")
    lang = xp(".//tei:lang/text()" if ns else ".//lang/text()")

    return {
        #Compter le nombre d'occurrences des éléments du niveau 2 parallèle (à commenter ou décommenter)
        #"definition": len(definition),
        #"traduction": len(traduction),
        #"exemple": len(exemple),
        #"renvoi": len(renvoi),
        #"reference": len(reference),
        #"encycl": len(encycl),
        #"etym": len(etym),
        #"glose": len(glose),

        #Compter le nombre d'occurrences des éléments du niveau 3 parallèle (à commenter ou décommenter)
        "bibl": len(bibl),
        "auteur": len(auteur),
        "oRef": len(oRef),
        "foreign": len(foreign),
        "lang": len(lang),

    }


# =====================================================
# 2. SCORING ENTRIES
# =====================================================

def compute_scores(alignment):
    tp = len(alignment["matches"])
    fp = len(alignment["insertions"]) + len(alignment["replacements"])
    fn = len(alignment["deletions"]) + len(alignment["replacements"])

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if (precision + recall) else 0,
        "pred_count": tp + fp,
        "gold_count": tp + fn,
    }


# =====================================================
# 3. SCORING BALISES GLOBALES
# =====================================================

def compute_tag_scores(pred_tags, gold_tags):
    scores = {}

    for tag in pred_tags.keys():
        pred = pred_tags[tag]
        gold = gold_tags[tag]

        tp = min(pred, gold)
        fp = max(0, pred - gold)
        fn = max(0, gold - pred)

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

        scores[tag] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "pred_count": pred,
            "gold_count": gold
        }

    return scores


# =====================================================
# 4. PIPELINE GLOBAL
# =====================================================

def evaluation(pred_path, gold_path, output_path):
    pred_xml = Path(pred_path).read_text(encoding="utf-8")
    gold_xml = Path(gold_path).read_text(encoding="utf-8")

    # --- TAGS ---
    pred_tags = extract_global_tags(pred_xml)
    gold_tags = extract_global_tags(gold_xml)

    # --- FINAL REPORT ---
    final_report = {
        "tags": compute_tag_scores(pred_tags, gold_tags)
    }

    # --- SAVE ---
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    return final_report

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

# Pour chaque document x prompt x modele, on produit un fichier JSON d'evaluation dans eval_dir, sans jamais arrêter le traitement en cas de fichier manquant ou mal formé.
# Dans les paramètres de la fonction, on décide quel est le niveau de la vérité de terrain (chemin relatif) à partir de laquelle on va obtenir les métriques souhaitées.
def traiter_niveau(niveau_label, prompts_dir, output_dir, eval_dir, gt_dir="gt/niveau3", inputs_dir="input"):
    Path(eval_dir).mkdir(parents=True, exist_ok=True)

    prompts_n = sorted(Path(prompts_dir).glob("*.txt"))
    models_n = sorted([p for p in Path(output_dir).iterdir() if p.is_dir()])
    inputs_n = sorted(Path(inputs_dir).glob("*.txt"))

    print(f"\n########## NIVEAU: {niveau_label} ##########")
    print("prompts trouves:", [p.name for p in prompts_n])
    print("models trouves:", [m.name for m in models_n])
    print("inputs trouves:", [i.name for i in inputs_n])

    fichiers_mal_formes_n = []

    for input_file in inputs_n:
        doc_name = input_file.stem
        gt_file = Path(gt_dir) / f"{doc_name}.xml"
        print(f"\n=== [{niveau_label}] DOCUMENT: {doc_name} ===")

        for prompt in prompts_n:
            print(f"  -> PROMPT: {prompt.stem}")

            for model_dir in models_n:
                output_file = model_dir / prompt.stem / f"{doc_name}.xml"

                if not output_file.exists():
                    print(f"     -> {model_dir.name}: fichier manquant ({output_file}), on continue")
                    continue

                print(f"     -> {model_dir.name}")

                # Verification prealable : XML bien forme ?
                try:
                    ET.parse(output_file)
                except ET.ParseError as e:
                    message = f"     FICHIER MAL FORME (XML invalide) : {output_file} - {e}"
                    print(message)
                    fichiers_mal_formes_n.append(str(output_file))
                    continue

                # Evaluation, securisee
                try:
                    evaluation(
                        output_file,
                        gt_file,
                        output_path=str(Path(eval_dir) / f"{model_dir.name}_{prompt.stem}_{doc_name}.json")
                    )
                except Exception as e:
                    message = f"     ERREUR lors de l'evaluation de {output_file} : {e}"
                    print(message)
                    fichiers_mal_formes_n.append(str(output_file))
                    continue

    return {
        "niveau": niveau_label,
        "prompts": prompts_n,
        "models": models_n,
        "inputs": inputs_n,
        "eval_dir": eval_dir,
        "fichiers_mal_formes": fichiers_mal_formes_n,
    }


# --- Lancement pour le niveau souhaité de la fonction traiter_niveau() ---
config_niveau_parallele = traiter_niveau(
    niveau_label="niveau_parallele_3",
    prompts_dir="prompts/niveau_parallele_3",
    output_dir="output/niveau_parallele_3",
    eval_dir="evaluation/niveau_parallele_3",
)

# --- Récapitulatif final ---
total_mal_formes = len(config_niveau_parallele["fichiers_mal_formes"])
if total_mal_formes:
    print(f"\n\n{total_mal_formes} fichier(s) mal forme(s) ou en erreur au total, ignores pendant le traitement :")
    for f in config_niveau_parallele["fichiers_mal_formes"]:
        print(f"   - [{config_niveau_parallele['niveau']}] {f}")
else:
    print("\n\nAucun fichier mal forme detecte.")


########## NIVEAU: niveau_parallele_3 ##########
prompts trouves: ['FS-R.txt', 'FS.txt', 'ZS.txt']
models trouves: ['gemini-3.1-flash-lite-preview', 'gemma-3-27b-it', 'gpt-5-mini', 'gpt-5.4-mini']
inputs trouves: ['TR1_p2001-2002.txt', 'TR1_p453-454.txt', 'TR2_p1785-1786.txt', 'TR2_p37-38.txt', 'TR3_p5-6.txt', 'TR3_p7-8.txt', 'TR4_p131-132.txt', 'TR5_p489-490.txt', 'TR5_p505-506.txt', 'TR6_p1003-1004.txt']

=== [niveau_parallele_3] DOCUMENT: TR1_p2001-2002 ===
  -> PROMPT: FS-R
     -> gemini-3.1-flash-lite-preview
     FICHIER MAL FORME (XML invalide) : output/niveau_parallele_3/gemini-3.1-flash-lite-preview/FS-R/TR1_p2001-2002.xml - mismatched tag: line 7, column 737
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini
  -> PROMPT: FS
     -> gemini-3.1-flash-lite-preview
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini
  -> PROMPT: ZS
     -> gemini-3.1-flash-lite-preview
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini

=== [niveau_parallele_3

In [ ]:
from collections import defaultdict
import json
import pandas as pd

# =====================================================
# MICRO F-MESURE PAR MODELE, TYPE DE PROMPT ET NIVEAU
# =====================================================


def get_prompt_type(prompt_stem: str) -> str:
    return prompt_stem.split("_")[0]


def agreger_micro_f1(config):
    # Parcourt les JSON deja produits pour un niveau donne et retourne
    # les lignes agregees (une ligne par modele x type_prompt x tag).
    niveau_label = config["niveau"]
    eval_dir = Path(config["eval_dir"])
    prompts_n = config["prompts"]
    models_n = config["models"]
    inputs_n = config["inputs"]

    agg = defaultdict(lambda: defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0}))
    fichiers_ignores = []

    for input_file in inputs_n:
        doc_name = input_file.stem
        for prompt in prompts_n:
            prompt_type = get_prompt_type(prompt.stem)
            for model_dir in models_n:
                eval_file = eval_dir / f"{model_dir.name}_{prompt.stem}_{doc_name}.json"
                if not eval_file.exists():
                    continue

                try:
                    with open(eval_file, "r", encoding="utf-8") as f:
                        report = json.load(f)
                except (json.JSONDecodeError, OSError) as e:
                    print(f"  FICHIER D'EVALUATION ILLISIBLE, ignore : {eval_file} - {e}")
                    fichiers_ignores.append(str(eval_file))
                    continue

                key = (model_dir.name, prompt_type)
                for tag, scores in report.get("tags", {}).items():
                    pred = scores["pred_count"]
                    gold = scores["gold_count"]
                    # tp=min(pred,gold) ; fp=max(0,pred-gold) ; fn=max(0,gold-pred)
                    # (coherent avec compute_tag_scores)
                    tp = min(pred, gold)
                    fp = max(0, pred - gold)
                    fn = max(0, gold - pred)
                    agg[key][tag]["tp"] += tp
                    agg[key][tag]["fp"] += fp
                    agg[key][tag]["fn"] += fn

    rows = []
    for (model, prompt_type), tags in agg.items():
        for tag, counts in tags.items():
            tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
            rows.append({
                "modele": model,
                "type_prompt": prompt_type,
                "niveau": niveau_label,
                "tag": tag,
                "tp": tp, "fp": fp, "fn": fn,
                "precision_micro": round(precision, 4),
                "recall_micro": round(recall, 4),
                "f1_micro": round(f1, 4),
            })

    if fichiers_ignores:
        print(f"\n  [{niveau_label}] {len(fichiers_ignores)} fichier(s) d'evaluation ignore(s) (JSON illisible).")

    return rows


# --- Agregation, puis lancement ---
toutes_les_lignes = agreger_micro_f1(config_niveau_parallele)

#Dataframe avec toutes ces informations
df_micro = pd.DataFrame(toutes_les_lignes).sort_values(
    ["modele", "type_prompt", "niveau", "tag"]
).reset_index(drop=True)

eval_dir = Path(config_niveau_parallele["eval_dir"])
output_csv =  eval_dir/ "micro_f1_par_modele_prompt_niveau.csv"
df_micro.to_csv(output_csv, index=False)
print(f"Resultats exportes vers : {output_csv}")

df_micro

Resultats exportes vers : evaluation/niveau_parallele_3/micro_f1_par_modele_prompt_niveau.csv


,modele,type_prompt,niveau,tag,tp,fp,fn,precision_micro,recall_micro,f1_micro
0,gemini-3.1-flash-lite-preview,FS,niveau_parallele_3,auteur,109,1,19,0.9909,0.8516,0.9160
1,gemini-3.1-flash-lite-preview,FS,niveau_parallele_3,bibl,44,0,44,1.0000,0.5000,0.6667
2,gemini-3.1-flash-lite-preview,FS,niveau_parallele_3,foreign,11,21,2,0.3438,0.8462,0.4889
3,gemini-3.1-flash-lite-preview,FS,niveau_parallele_3,lang,5,0,6,1.0000,0.4545,0.6250
4,gemini-3.1-flash-lite-preview,FS,niveau_parallele_3,oRef,18,4,10,0.8182,0.6429,0.7200
5,gemini-3.1-flash-lite-preview,FS-R,niveau_parallele_3,auteur,92,2,17,0.9787,0.8440,0.9064
6,gemini-3.1-flash-lite-preview,FS-R,niveau_parallele_3,bibl,38,0,34,1.0000,0.5278,0.6909
7,gemini-3.1-flash-lite-preview,FS-R,niveau_parallele_3,foreign,3,24,4,0.1111,0.4286,0.1765
8,gemini-3.1-flash-lite-preview,FS-R,niveau_parallele_3,lang,5,0,5,1.0000,0.5000,0.6667
9,gemini-3.1-flash-lite-preview,FS-R,niveau_parallele_3,oRef,18,9,1,0.6667,0.9474,0.7826


In [ ]:
# --- Vue pivotée du premier dataframe : micro F-mesure par tag, pour chaque (modele, type de prompt et niveau) ---
pivot_f1 = df_micro.pivot_table(
    index=["modele", "type_prompt", "niveau"],
    columns="tag",
    values="f1_micro"
).round(4)
output_pivot =  eval_dir/ "micro_f1_simplifie.csv"
pivot_f1.to_csv(output_pivot, index=False)
print(f"Resultats exportes vers : {output_pivot}")

pivot_f1

Resultats exportes vers : evaluation/niveau_parallele_3/micro_f1_simplifie.csv


tag                                                           auteur    bibl  \
modele                        type_prompt niveau                               
gemini-3.1-flash-lite-preview FS          niveau_parallele_3  0.9160  0.6667   
                              FS-R        niveau_parallele_3  0.9064  0.6909   
                              ZS          niveau_parallele_3  0.8342  0.3922   
gemma-3-27b-it                FS          niveau_parallele_3  0.3350  0.3956   
                              FS-R        niveau_parallele_3  0.4579  0.4474   
                              ZS          niveau_parallele_3  0.2174  0.4403   
gpt-5-mini                    FS          niveau_parallele_3  0.5641  0.3576   
                              FS-R        niveau_parallele_3  0.4115  0.5116   
                              ZS          niveau_parallele_3  0.0000  0.4855   
gpt-5.4-mini                  FS          niveau_parallele_3  0.1040  0.4390   
                              FS-R        niveau_parallele_3  0.0000  0.4855   
                              ZS          niveau_parallele_3  0.1040  0.4390   

tag                                                           foreign    lang  \
modele                        type_prompt niveau                                
gemini-3.1-flash-lite-preview FS          niveau_parallele_3   0.4889  0.6250   
                              FS-R        niveau_parallele_3   0.1765  0.6667   
                              ZS          niveau_parallele_3   0.5882  0.3333   
gemma-3-27b-it                FS          niveau_parallele_3   0.0000  0.1333   
                              FS-R        niveau_parallele_3   0.0000  0.0000   
                              ZS          niveau_parallele_3   0.0000  0.0000   
gpt-5-mini                    FS          niveau_parallele_3   0.3000  0.7273   
                              FS-R        niveau_parallele_3   0.3000  0.4444   
                              ZS          niveau_parallele_3   0.0000  0.0000   
gpt-5.4-mini                  FS          niveau_parallele_3   0.0000  0.0000   
                              FS-R        niveau_parallele_3   0.0000  0.0000   
                              ZS          niveau_parallele_3   0.0000  0.0000   

tag                                                             oRef  
modele                        type_prompt niveau                      
gemini-3.1-flash-lite-preview FS          niveau_parallele_3  0.7200  
                              FS-R        niveau_parallele_3  0.7826  
                              ZS          niveau_parallele_3  0.6809  
gemma-3-27b-it                FS          niveau_parallele_3  0.0541  
                              FS-R        niveau_parallele_3  0.0000  
                              ZS          niveau_parallele_3  0.0000  
gpt-5-mini                    FS          niveau_parallele_3  0.2857  
                              FS-R        niveau_parallele_3  0.2439  
                              ZS          niveau_parallele_3  0.0000  
gpt-5.4-mini                  FS          niveau_parallele_3  0.0000  
                              FS-R        niveau_parallele_3  0.0000  
                              ZS          niveau_parallele_3  0.0000